In [1]:

from pathlib import Path
import pandas as pd

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "notebooks" else cwd

ANON_PATH = PROJECT_ROOT / "data" / "anonymized" / "level1" / "adult_level1.csv"

if not ANON_PATH.exists():
    raise FileNotFoundError(
        f"Anonymized dataset not found: {ANON_PATH}\n"
        "Update ANON_PATH to the actual CSV location."
    )

df_level1 = pd.read_csv(
    ANON_PATH,
    skipinitialspace=True,
    dtype={"zip_code": "string"}
)

# Normalize column names
df_level1.columns = df_level1.columns.str.strip()

print("Dataset shape:", df_level1.shape)
print("Income column present:", "income" in df_level1.columns)
print("Age examples:", df_level1["age"].head().tolist())
print("ZIP examples:", df_level1["zip_code"].head().tolist())

assert "income" in df_level1.columns
assert df_level1["income"].notna().all()

print("\nDay 10 dataset loaded successfully.")


Dataset shape: (32561, 16)
Income column present: True
Age examples: ['30-39', '50-59', '30-39', '50-59', '20-29']
ZIP examples: ['938**', '245**', '132**', '460**', '420**']

Day 10 dataset loaded successfully.


In [2]:
# Split the data and train Logistic Regression


from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Separate features and target
X = df_level1.drop(columns=["income"])
y = df_level1["income"].astype(str).str.strip()

# Ensure generalized values are categorical
X["age"] = X["age"].astype("string")
X["zip_code"] = X["zip_code"].astype("string")

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

# Same split settings as baseline
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

# Fit preprocessing only on training data
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Train Logistic Regression with baseline settings
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train_processed, y_train)
y_pred = model.predict(X_test_processed)

# Evaluate the positive class (>50K)
day10_result = {
    "experiment": "Level 1",
    "day": 10,
    "model": "Logistic Regression",
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(
        y_test, y_pred, pos_label=">50K", zero_division=0
    ),
    "recall": recall_score(
        y_test, y_pred, pos_label=">50K", zero_division=0
    ),
    "f1_score": f1_score(
        y_test, y_pred, pos_label=">50K", zero_division=0
    )
}

display(pd.DataFrame([day10_result]))
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Processed features:", X_train_processed.shape[1])


,experiment,day,model,accuracy,precision,recall,f1_score
0,Level 1,10,Logistic Regression,0.85552,0.730345,0.633929,0.67873


Training rows: 26048
Test rows: 6513
Processed features: 1013


In [3]:
# Save the Day 10 result


RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

output_path = RESULTS_DIR / "day10_level1_logistic_regression.csv"

pd.DataFrame([day10_result]).to_csv(output_path, index=False)

print("Day 10 results saved to:", output_path)


Day 10 results saved to: d:\VS Code\Projects\python\Internship\results\day10_level1_logistic_regression.csv
